First, I prepared the data by loading it, doing the chronological ordering, and splitting it.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data exported from Week 5.
regression_df = pd.read_csv(
    "regression_data.csv",
    parse_dates=["hour_local"]
)

# All remaining columns are predictors.
core_features = regression_df.columns.drop(
    ["hour_local", "avg_departure_delay"]
).tolist()

print(f"Loaded {len(regression_df):,} rows and {len(core_features)} predictors.")

Loaded 28,499 rows and 23 predictors.


In [2]:
# Sort our data by date and time
data = regression_df.sort_values("hour_local")

# X contains the features we use to make predictions
X = data[core_features]

# y contains the variable we want to predict
y = data["avg_departure_delay"]

In [4]:
# Sort data chronologically
data = regression_df.sort_values("hour_local")

X = data[core_features]
y = data["avg_departure_delay"]

In [5]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Split into training, validation, and testing data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

#Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

In [6]:
print("Training rows (70%):", len(X_train))
print("Validation rows (15%):", len(X_val))
print("Testing rows (15%):", len(X_test))

Training rows (70%): 19949
Validation rows (15%): 4275
Testing rows (15%): 4275


Then, I decided to try recursive feature elimination (RFE) on linear regression. When you give RFE a number of features you want to include in your model, it creates a model with all features, determine which feature is influencing the model the least, and drops that feature iteratively until it reaches the amount of features you want. Since we don't know the optimal amount of features, I had it go through a range of 3-23, since we have 23 features, and saw which model it created for each had the best r2 score.

In [17]:
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

#Making the linear regression model
linear_regression = LinearRegression()

#Trying recursive feature elimination for different amounts of features
for i in range (3, 24):
  rfe = RFE(estimator=linear_regression, n_features_to_select=i, step=1)
  rfe.fit(X_train_scaled, y_train)

  #Using the validation data to make predictions
  predictions = rfe.predict(X_val_scaled)

  #Seeing how well the model created by rfe performs on the validation data
  r2 = r2_score(y_val, predictions)
  rmse = root_mean_squared_error(y_val, predictions)
  mae = mean_absolute_error(y_val, predictions)

  #Printing the results for our reference
  print(f"{i}{" features: r2 is "} {r2}")

3 features: r2 is  0.01621623446706233
4 features: r2 is  0.029778431618073586
5 features: r2 is  0.054470252289297694
6 features: r2 is  0.06476858031200194
7 features: r2 is  0.07205488210263988
8 features: r2 is  0.07937378020459529
9 features: r2 is  0.07802597604560912
10 features: r2 is  0.08011507113322591
11 features: r2 is  0.0818662096467263
12 features: r2 is  0.08291815491732613
13 features: r2 is  0.08464620657127753
14 features: r2 is  0.08411440592762298
15 features: r2 is  0.0842041424176041
16 features: r2 is  0.08458862303274206
17 features: r2 is  0.08474255882081916
18 features: r2 is  0.08442457941204129
19 features: r2 is  0.08488711133607219
20 features: r2 is  0.08487759136480877
21 features: r2 is  0.08498094434263603
22 features: r2 is  0.08493924111031892
23 features: r2 is  0.08494800367381095


The linear regression model found by RFE that has the best r2 score is the one that uses 21 features, with an r2 score of 0.084981. I then found the features included in this model so I could recreate the model:

In [58]:
rfe = RFE(estimator=linear_regression, n_features_to_select=21, step=1)
rfe.fit(X_train_scaled, y_train)

selected_features = X_train.columns[rfe.support_]
print(selected_features)

Index(['scheduled_flights', 'airlines', 'origins', 'avg_distance',
       'avg_scheduled_flight_time', 'temperature_c', 'wind_max_ms',
       'visibility_min_m', 'dew_point_spread_c', 'freezing_deficit_c',
       'high_wind', 'traffic_low_visibility', 'hour_sin', 'hour_cos',
       'season_sin', 'season_cos', 'dow_tue', 'dow_wed', 'dow_fri', 'dow_sat',
       'dow_sun'],
      dtype='object')


Now, the model can be retrained using both the validation and training data, since we have its official validation score:

In [68]:
#Make linear regression model given by these features:
X_train_selected_LR = X_train[selected_features]
X_test_selected_LR = X_test[selected_features]
X_val_selected_LR = X_val[selected_features]

#Combine the validation and training data to train this last model:
X_train_val_LR = pd.concat([X_train_selected_LR, X_val_selected_LR])
y_train_val_LR = pd.concat([y_train, y_val])

#Fit
rfe_linear_regression=LinearRegression()
rfe_linear_regression.fit(X_train_val_LR, y_train_val_LR)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


Now we have a linear regression model chosen by rfe that has a validation r2 of 0.084878.

Next, I repeated the process with ridge regression:

In [64]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

#Trying recursive feature elimination for different amounts of features
for i in range (3, 24):
  rfe = RFE(estimator=ridge, n_features_to_select=i, step=1)
  rfe.fit(X_train_scaled, y_train)

  #Using the validation data to make predictions
  predictions = rfe.predict(X_val_scaled)

  #Seeing how well the model created by rfe performs on the validation data
  r2 = r2_score(y_val, predictions)
  rmse = root_mean_squared_error(y_val, predictions)
  mae = mean_absolute_error(y_val, predictions)

  #Printing the results for our reference
  print(f"{i}{" features: r2 is "} {r2}")

3 features: r2 is  0.016226992842600763
4 features: r2 is  0.0297877303001437
5 features: r2 is  0.05448015957705077
6 features: r2 is  0.06477801587412768
7 features: r2 is  0.07206437132261823
8 features: r2 is  0.07938383184525999
9 features: r2 is  0.07803602000885568
10 features: r2 is  0.08012524812118593
11 features: r2 is  0.08187620755627478
12 features: r2 is  0.08292980846347109
13 features: r2 is  0.08465769802085599
14 features: r2 is  0.08412613120002421
15 features: r2 is  0.08421593452427945
16 features: r2 is  0.08459989477561725
17 features: r2 is  0.08475385942480529
18 features: r2 is  0.08443379826782749
19 features: r2 is  0.0848963105722863
20 features: r2 is  0.08488671939636461
21 features: r2 is  0.08499008124143947
22 features: r2 is  0.08494833719720962
23 features: r2 is  0.08495715793418435


The ridge regression model found by RFE that has the best r2 score is the one that uses 21 features, with an r2 score of 0.0849901. I then found the features included in this model so I could recreate the model:

In [65]:
rfe = RFE(estimator=ridge, n_features_to_select=21, step=1)
rfe.fit(X_train_scaled, y_train)

selected_features = X_train.columns[rfe.support_]
print(selected_features)

Index(['scheduled_flights', 'airlines', 'origins', 'avg_distance',
       'avg_scheduled_flight_time', 'temperature_c', 'wind_max_ms',
       'visibility_min_m', 'dew_point_spread_c', 'freezing_deficit_c',
       'high_wind', 'traffic_low_visibility', 'hour_sin', 'hour_cos',
       'season_sin', 'season_cos', 'dow_tue', 'dow_wed', 'dow_fri', 'dow_sat',
       'dow_sun'],
      dtype='object')


Here is the retraining of the model using both validation and training data:

In [66]:
#Make linear regression model given by these features:
X_train_selected_R = X_train[selected_features]
X_test_selected_R = X_test[selected_features]
X_val_selected_R = X_val[selected_features]

#Combine the validation and training data to train this last model:
X_train_val_R = pd.concat([X_train_selected_R, X_val_selected_R])
y_train_val_R = pd.concat([y_train, y_val])

#Fit
rfe_ridge=Ridge(alpha=1.0)
rfe_ridge.fit(X_train_val_R, y_train_val_R)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


Now we have a ridge regression model chosen by rfe that has a validation r2 of 0.0849901.

Last is lasso regression:

In [53]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0, max_iter=10000)

#Trying recursive feature elimination for different amounts of features
for i in range (3, 24):
  rfe = RFE(estimator=lasso, n_features_to_select=i, step=1)
  rfe.fit(X_train_scaled, y_train)

  #Using the validation data to make predictions
  predictions = rfe.predict(X_val_scaled)

  #Seeing how well the model created by rfe performs on the validation data
  r2 = r2_score(y_val, predictions)
  rmse = root_mean_squared_error(y_val, predictions)
  mae = mean_absolute_error(y_val, predictions)

  #Printing the results for our reference
  print(f"{i}{" features: r2 is "} {r2}")

3 features: r2 is  0.04696785916992219
4 features: r2 is  0.05596391986842175
5 features: r2 is  0.0617832607872677
6 features: r2 is  0.0684517087456652
7 features: r2 is  0.06859124113642323
8 features: r2 is  0.06946712098248786
9 features: r2 is  0.0702192532491025
10 features: r2 is  0.070214033397483
11 features: r2 is  0.070214033397483
12 features: r2 is  0.070214033397483
13 features: r2 is  0.070214033397483
14 features: r2 is  0.070214033397483
15 features: r2 is  0.070214033397483
16 features: r2 is  0.070214033397483
17 features: r2 is  0.070214033397483
18 features: r2 is  0.070214029843369
19 features: r2 is  0.07021403550604344
20 features: r2 is  0.07021403550604344
21 features: r2 is  0.07021403620046818
22 features: r2 is  0.07021403620046818
23 features: r2 is  0.07021403721504904


The lasso regression model found by RFE that has the best r2 score is the one that uses all 23 features, with an r2 score of 0.070214037. Since it uses all features, the model can be found immediately:

In [62]:
#Combine the validation and training data to train this last model:
X_train_val_L = pd.concat([X_train, X_val])
y_train_val_L = pd.concat([y_train, y_val])

#Fit
rfe_lasso=Lasso(alpha=1.0, max_iter=1000)
rfe_lasso.fit(X_train_val_L, y_train_val_L)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


Using RFE, the following models were found:
- Linear regression: 21 features, r2 of 0.084878
- Ridge regression: 21 features, r2 of 0.0849901
- Lasso regression: 23 features, r2 of 0.070214037

Thus, the best-performing model found is the ridge regression model. So, I then tested this officially on the testing data for the final metrics of the model's performance:

In [ ]:
ridge_predictions = rfe_ridge.predict(X_test_selected_R)
r2 = r2_score(y_test, ridge_predictions)
rmse = root_mean_squared_error(y_test, ridge_predictions)
mae = mean_absolute_error(y_test, ridge_predictions)
print(f"{"r2: "} {r2}")
print(f"{"RMSE: "} {rmse}")
print(f"{"MAE"} {mae}")

So, the ridge regression model with 21 features has an r2 score of 0.09965, an RMSE of 28.973, and an MAE of 13.650 when tested with the testing data.

I then tried some random forest models:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

#Split into training, validation, and testing data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)


for i in range (1, 101):
    for j in range (1, 9):
        forest_model = RandomForestRegressor(
            n_estimators=i,
            max_depth=j,
            random_state=42
        )

        # Train model
        forest_model.fit(X_train, y_train)

        # Make validation predictions
        forest_pred = forest_model.predict(X_val)

        r2 = r2_score(y_val, predictions)
        print(f"{"Depth of " }{j}{" with "}{i}{" estimators: r2 is "} {r2}")